# Cyberbullying Classification: Support Vector Machines (LinearSVC)

## 1. Academic Introduction & Mathematical Foundations
Support Vector Machines (SVM) represent one of the most effective classical machine learning algorithms for text classification. In high-dimensional sparse feature spaces (such as TF-IDF spaces where $d = 10,000$), documents are nearly always linearly separable.

### Maximum Margin Hyperplane Formulation
Given training pairs $(\mathbf{x}_i, y_i)$ with $y_i \in \{-1, +1\}$, Linear SVM solves the primal optimization problem:

$$\min_{\mathbf{w}, b, \boldsymbol{\xi}} \frac{1}{2} \|\mathbf{w}\|^2 + C \sum_{i=1}^N \xi_i$$
$$\text{subject to } y_i (\mathbf{w}^T \mathbf{x}_i + b) \ge 1 - \xi_i, \quad \xi_i \ge 0, \quad \forall i$$

where:
* $\frac{2}{\|\mathbf{w}\|}$ is the geometric margin separating the two classes.
* $\xi_i$ are slack variables representing margin violations (soft-margin formulation).
* $C > 0$ controls the penalty for misclassifications (regularization parameter).

### Loss Function: Hinge Loss vs. Log-Loss
Unlike Logistic Regression, which optimizes the smooth convex log-loss:
$$\mathcal{L}_{\text{log}}(y, f(\mathbf{x})) = \log(1 + e^{-y f(\mathbf{x})})$$
Linear SVM minimizes the piecewise-linear **hinge loss**:
$$\mathcal{L}_{\text{hinge}}(y, f(\mathbf{x})) = \max(0, 1 - y f(\mathbf{x}))$$

This produces a sparse dual representation where only training instances lying on or within the margin (**support vectors**) determine the final decision boundary $\mathbf{w}$.

### Probability Calibration via Platt Scaling
Because standard SVM yields uncalibrated geometric distances $f(\mathbf{x}) = \mathbf{w}^T \mathbf{x} + b$, we wrap the model with **Platt Scaling** (`CalibratedClassifierCV`) via an internal sigmoid transformation:
$$P(y = 1 \mid f(\mathbf{x})) = \frac{1}{1 + e^{A f(\mathbf{x}) + B}}$$
providing calibrated probabilities for risk scoring in our web moderation service.

In [ ]:
# Environment Configuration and Scientific Computing Imports
import os
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

LABEL_MAP = {0: 'Non-Bullying', 1: 'Bullying'}
os.makedirs(os.path.join("models", "svm"), exist_ok=True)

print("Environment and SVM module initialized.")

## 2. Ingestion of Preprocessed Partitions & Vectorization
We load the partitioned datasets and transform them using our pre-fitted `models/tfidf_vectorizer.joblib`.

In [ ]:
# Load preprocessed partitions and vectorizer
train_path = os.path.join("data", "processed_train.csv")
test_path = os.path.join("data", "processed_test.csv")
vectorizer_path = os.path.join("models", "tfidf_vectorizer.joblib")

if not os.path.exists(train_path) or not os.path.exists(vectorizer_path):
    raise FileNotFoundError("Prerequisite data or vectorizer not found. Please execute Notebook 02 first.")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
tfidf_vectorizer = joblib.load(vectorizer_path)

X_train = tfidf_vectorizer.transform(train_df['clean_text'].fillna(''))
X_test = tfidf_vectorizer.transform(test_df['clean_text'].fillna(''))

y_train = train_df['label'].values
y_test = test_df['label'].values

print(f"Loaded X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")

## 3. SVM Hyperparameter Optimization ($C$)
We evaluate the regularization penalty $C \in \{0.01, 0.1, 1.0, 5.0, 10.0\}$ using 5-Fold Stratified Cross-Validation with squared hinge loss (`loss='squared_hinge'`) and balanced class weighting.

In [ ]:
# Cross-Validation for LinearSVC Regularization Parameter C
c_candidates = [0.01, 0.1, 1.0, 5.0, 10.0]
cv_scores = []
cv_stds = []

cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Evaluating LinearSVC C hyperparameter via 5-Fold Stratified Cross-Validation:")
for c in c_candidates:
    svm_cand = LinearSVC(
        C=c,
        loss='squared_hinge',
        penalty='l2',
        class_weight='balanced',
        random_state=RANDOM_STATE,
        max_iter=2000
    )
    scores = cross_val_score(svm_cand, X_train, y_train, cv=cv_strat, scoring='f1', n_jobs=-1)
    cv_scores.append(scores.mean())
    cv_stds.append(scores.std())
    print(f"  C = {c:<5} | Mean F1-Score: {scores.mean():.4f} (+/- {scores.std():.4f})")

best_idx = np.argmax(cv_scores)
best_c = c_candidates[best_idx]
print(f"\nOptimal SVM Regularization Parameter: C = {best_c} (F1 = {cv_scores[best_idx]:.4f})")

## 4. Model Training and Platt Probability Calibration
We fit the optimal `LinearSVC` base model and calibrate its decision margins using `CalibratedClassifierCV` via 5-fold cross-validation.

In [ ]:
# Fit base LinearSVC model and calibrated wrapper
start_time = time.time()

base_svm = LinearSVC(
    C=best_c,
    loss='squared_hinge',
    penalty='l2',
    class_weight='balanced',
    random_state=RANDOM_STATE,
    max_iter=2000
)

# Platt Scaling probability calibration
calibrated_svm = CalibratedClassifierCV(estimator=base_svm, cv=5, method='sigmoid')
calibrated_svm.fit(X_train, y_train)

# Also fit standalone base_svm to inspect raw hyperplane weights
base_svm.fit(X_train, y_train)

train_duration = time.time() - start_time
print(f"SVM training and Platt calibration completed in {train_duration:.3f} seconds.")

## 5. Quantitative Performance Evaluation on Hold-Out Test Set
We compute comprehensive classification metrics and print the detailed classification report.

In [ ]:
# Generate Predictions and Calibrated Probabilities
y_pred = calibrated_svm.predict(X_test)
y_prob = calibrated_svm.predict_proba(X_test)[:, 1]

# Quantitative Metric Computation
acc = accuracy_score(y_test, y_pred)
prec_macro = precision_score(y_test, y_pred, average='macro')
rec_macro = recall_score(y_test, y_pred, average='macro')
f1_macro = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')
roc_auc = roc_auc_score(y_test, y_prob)

prec_bully = precision_score(y_test, y_pred, pos_label=1)
rec_bully = recall_score(y_test, y_pred, pos_label=1)
f1_bully = f1_score(y_test, y_pred, pos_label=1)

print("=== SVM Classification Report (Hold-Out Test Set) ===")
print(classification_report(y_test, y_pred, target_names=['Non-Bullying (0)', 'Bullying (1)'], digits=4))

# Store SVM Metrics
svm_metrics = {
    'Model': 'TF-IDF + Linear SVM',
    'Accuracy': round(acc, 4),
    'Macro Precision': round(prec_macro, 4),
    'Macro Recall': round(rec_macro, 4),
    'Macro F1': round(f1_macro, 4),
    'Weighted F1': round(f1_weighted, 4),
    'Bullying Precision': round(prec_bully, 4),
    'Bullying Recall': round(rec_bully, 4),
    'Bullying F1': round(f1_bully, 4),
    'ROC-AUC': round(roc_auc, 4),
    'Training Time (s)': round(train_duration, 4)
}

print("\n=== SVM Metric Summary ===")
for k, v in svm_metrics.items():
    print(f"  {k:<22}: {v}")

## 6. Confusion Matrix & ROC Visualizations
We plot the absolute and normalized confusion matrices along with the ROC and Precision-Recall trajectories.

In [ ]:
# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Absolute Counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', cbar=False, ax=axes[0],
            xticklabels=['Non-Bullying', 'Bullying'],
            yticklabels=['Non-Bullying', 'Bullying'],
            annot_kws={'size': 13, 'weight': 'bold'})
axes[0].set_title("Linear SVM: Confusion Matrix (Counts)", pad=12, fontweight='bold')
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Ground Truth")

# Subplot 2: Normalized Recall
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='YlOrBr', cbar=False, ax=axes[1],
            xticklabels=['Non-Bullying', 'Bullying'],
            yticklabels=['Non-Bullying', 'Bullying'],
            annot_kws={'size': 13, 'weight': 'bold'})
axes[1].set_title("Linear SVM: Normalized Recall Matrix", pad=12, fontweight='bold')
axes[1].set_xlabel("Predicted Label")
axes[1].set_ylabel("True Ground Truth")

plt.tight_layout()
plt.show()

## 7. Comparative Benchmark: Linear SVM vs. Logistic Regression
We load the evaluation metrics recorded in `models/metrics_logistic_regression.json` and construct a side-by-side comparative table and visualization.

In [ ]:
# Ingest Logistic Regression Metrics
lr_metrics_path = os.path.join("models", "metrics_logistic_regression.json")
if os.path.exists(lr_metrics_path):
    with open(lr_metrics_path, 'r', encoding='utf-8') as f:
        lr_metrics = json.load(f)
else:
    lr_metrics = {'Model': 'Logistic Regression (Baseline)', 'Accuracy': 0.90, 'Macro F1': 0.89, 'Bullying F1': 0.88, 'ROC-AUC': 0.95}

# Comparative DataFrame
comparison_df = pd.DataFrame([lr_metrics, svm_metrics])
display_metrics = ['Model', 'Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1', 'Bullying Precision', 'Bullying Recall', 'Bullying F1', 'ROC-AUC']
print("=== Baseline Linear Models Comparison Table ===")
print(comparison_df[[c for c in display_metrics if c in comparison_df.columns]].to_string(index=False))

# Visualization: Grouped Bar Chart
metrics_to_plot = ['Accuracy', 'Bullying Precision', 'Bullying Recall', 'Bullying F1', 'ROC-AUC']
lr_vals = [lr_metrics.get(m, 0) for m in metrics_to_plot]
svm_vals = [svm_metrics.get(m, 0) for m in metrics_to_plot]

x = np.arange(len(metrics_to_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5.5))
rects1 = ax.bar(x - width/2, lr_vals, width, label='Logistic Regression', color='#2b5c8f', edgecolor='black', alpha=0.85)
rects2 = ax.bar(x + width/2, svm_vals, width, label='Linear SVM', color='#e67e22', edgecolor='black', alpha=0.85)

ax.set_ylabel('Score (0.0 to 1.0)')
ax.set_title('Comparative Benchmark: Logistic Regression vs. Linear SVM', pad=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot, fontsize=11, fontweight='bold')
ax.set_ylim(0.70, 1.02)
ax.legend(frameon=True, loc='lower right')

for rect in list(rects1) + list(rects2):
    height = rect.get_height()
    ax.annotate(f'{height:.3f}',
                xy=(rect.get_x() + rect.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Feature Importance via Normal Vector to the Hyperplane
The normal vector $\mathbf{w}$ represents the orientation of the separating hyperplane. We extract and plot the top 20 positive and negative hyperplane weights.

In [ ]:
# Extract Hyperplane Normal Vector Weights
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
svm_weights = base_svm.coef_[0]

top_pos_idx = np.argsort(svm_weights)[::-1][:20]
top_neg_idx = np.argsort(svm_weights)[:20]

top_pos_features = feature_names[top_pos_idx]
top_pos_weights = svm_weights[top_pos_idx]

top_neg_features = feature_names[top_neg_idx]
top_neg_weights = svm_weights[top_neg_idx]

# Plot Hyperplane Feature Weights
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].barh(range(len(top_pos_features)), top_pos_weights, color='#e67e22', edgecolor='black', alpha=0.85)
axes[0].set_yticks(range(len(top_pos_features)))
axes[0].set_yticklabels(top_pos_features, fontsize=11)
axes[0].invert_yaxis()
axes[0].set_title("Top 20 SVM Bullying Hyperplane Weights ($w > 0$)", pad=10, fontweight='bold')
axes[0].set_xlabel("Support Vector Weight")

axes[1].barh(range(len(top_neg_features)), np.abs(top_neg_weights), color='#2b5c8f', edgecolor='black', alpha=0.85)
axes[1].set_yticks(range(len(top_neg_features)))
axes[1].set_yticklabels(top_neg_features, fontsize=11)
axes[1].invert_yaxis()
axes[1].set_title("Top 20 SVM Non-Bullying Hyperplane Weights ($|w|, w < 0$)", pad=10, fontweight='bold')
axes[1].set_xlabel("Absolute Support Vector Weight")

plt.tight_layout()
plt.show()

## 9. Model Persistence & Artifact Export
We serialize the trained calibrated SVM model to `models/svm/model.joblib` and save evaluation metrics to `models/metrics_svm.json`.

In [ ]:
# Persist SVM Model Artifact
svm_export_path = os.path.join("models", "svm", "model.joblib")
joblib.dump(calibrated_svm, svm_export_path)
print(f"Calibrated Linear SVM saved to: '{svm_export_path}' ({os.path.getsize(svm_export_path)/1024:.1f} KB)")

# Export Metrics to JSON
metrics_export_path = os.path.join("models", "metrics_svm.json")
with open(metrics_export_path, 'w', encoding='utf-8') as f:
    json.dump(svm_metrics, f, indent=2)

print(f"SVM metrics successfully exported to: '{metrics_export_path}'")